[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_21_rope_cached.ipynb)

# 🔴 Hard: RoPE with a KV Cache

*Inference & Decoding*
RoPE, but the tokens you are rotating do not start at position 0.

Problem 24 hardcodes `pos = jnp.arange(T)` and rotates `q` and `k` with the
same table, which quietly requires `seq_q == seq_k` and a sequence that starts
at the beginning. Neither holds once there is a KV cache — and that is the
only situation where RoPE actually gets used at inference.

### Signature
```python
def rope_cached_attention(q, k_new, v_new, cache=None, base=10000.0):
    ...
```

| | shape | |
|---|---|---|
| `q`, `k_new`, `v_new` | `(B, H, seq_q, D)` | `D` even |
| `cache` | `None` or `(k_cached, v_cached)`, each `(B, H, seq_past, D)` | **`k_cached` is already rotated** |
| returns | `(out, (k_all, v_all))` | `out` is `(B, H, seq_q, D)` |

### Absolute positions, not 0..seq_q-1
RoPE encodes position by *rotating* each `(even, odd)` pair of the last axis by
an angle proportional to the token's index. Attention then recovers *relative*
position from the difference of the two angles — which only works if both sides
were rotated by their **absolute** slot in the sequence.

With `seq_past` tokens already cached, the incoming `seq_q` tokens occupy slots
`seq_past … seq_past + seq_q - 1`:

```python
seq_past = 0 if cache is None else cache[0].shape[-2]
positions = seq_past + jnp.arange(seq_q)     # 100 cached, 3 new -> [100, 101, 102]
```

The same vector goes to `q` and to `k_new`: they are the same slots.

### The cache is already rotated
A key is rotated once, when it is created, and stored that way. So
`k_cached` comes back rotated and you concatenate it **untouched**. Rotating
it a second time applies the angle twice and silently corrupts every past
position — no error, just a model that attends to the wrong places.

### The test that catches all of it
Run a sequence two ways: all at once, versus prefill-then-decode-one-token-at-a-time.
They must agree to floating-point tolerance. Get the position offset wrong, or
re-rotate the cache, and they will not.

The causal mask is the offset one you have already met:
`jnp.tril(jnp.ones((seq_q, seq_k), dtype=bool), k=seq_k - seq_q)`.

### RoPE itself, for reference
$$\theta_{t,j} = \frac{t}{\text{base}^{2j/D}}, \qquad j = 0 \dots D/2-1$$

Pair the last axis as `(0,1), (2,3), …` and rotate each pair by its angle:
$$(x_{2j}, x_{2j+1}) \mapsto
(x_{2j}\cos\theta - x_{2j+1}\sin\theta,\;
 x_{2j}\sin\theta + x_{2j+1}\cos\theta)$$

This is the same convention as problem 24, so with `cache=None` your rotation
must agree with `apply_rope` exactly.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def rope_cached_attention(q, k_new, v_new, cache=None, base=10000.0):
    """Causal attention with RoPE applied at absolute positions.

    Args:
        q, k_new, v_new: (B, H, seq_q, D), D even
        cache: None, or (k_cached, v_cached) each (B, H, seq_past, D).
               k_cached is ALREADY rotated — do not rotate it again.
        base: RoPE frequency base

    Returns:
        (out, (k_all, v_all)) with out of shape (B, H, seq_q, D)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

B, H, T, D = 1, 2, 8, 8
k = jax.random.split(jax.random.key(0), 3)
Q = jax.random.normal(k[0], (B, H, T, D))
K = jax.random.normal(k[1], (B, H, T, D))
V = jax.random.normal(k[2], (B, H, T, D))

full, _ = rope_cached_attention(Q, K, V)

out, cache = rope_cached_attention(Q[:, :, :5], K[:, :, :5], V[:, :, :5])
print("prefill 5:", out.shape, "cache:", cache[0].shape)
pieces = [out]
for t in range(5, T):
    print(f"  decoding slot {t} -> positions = [{t}]")
    out, cache = rope_cached_attention(
        Q[:, :, t:t + 1], K[:, :, t:t + 1], V[:, :, t:t + 1], cache)
    pieces.append(out)

stepwise = jnp.concatenate(pieces, axis=-2)
print("\nstepwise == all-at-once?",
      bool(jnp.allclose(stepwise, full, atol=1e-5)),
      f"(max diff {float(jnp.max(jnp.abs(stepwise - full))):.2e})")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("rope_cached")

# hint("rope_cached")      # stuck? nudge without the answer
# solution("rope_cached")  # spoiler: the reference implementation
# status()                 # your dashboard across all problems